# 09 Basic Polymer Analysis

This notebook is intentionally wired to use only preprocessed trajectories. Run `08_trajectory_preprocessing.ipynb` first so the GROMACS trajectory has been centered, compact-wrapped, and optionally fitted.

## Analysis input policy

Use `analysis_ready/step7_fitted.xtc` for alignment-sensitive analyses such as RMSD. Use `processed/step7_centered.xtc` when fitting would change the observable you want to inspect.

Do not use raw `step7_production.xtc` directly for polymer RMSD, Rg, or SASA. Raw periodic trajectories can contain broken molecules, box jumps, and whole-system drift.

## Centering and output groups

The centering group should usually be `center`. The output group should usually be `System`.

That combination keeps water and ions in the trajectory while keeping the polymer or selected complex centered for analysis.

In [ ]:
from pathlib import Path

SYSTEM_DIR = Path("../examples/output/md_tests/PHB4/gromacs/solvated_polymer").resolve()
ANALYSIS_TRAJECTORY = SYSTEM_DIR / "analysis_ready" / "step7_fitted.xtc"
CENTERED_TRAJECTORY = SYSTEM_DIR / "processed" / "step7_centered.xtc"
STRUCTURE = SYSTEM_DIR / "step7_production.tpr"
CENTER_INDEX = SYSTEM_DIR / "center.ndx"

analysis_inputs = {
    "analysis_trajectory": ANALYSIS_TRAJECTORY,
    "centered_trajectory": CENTERED_TRAJECTORY,
    "structure": STRUCTURE,
    "center_index": CENTER_INDEX,
}

for label, path in analysis_inputs.items():
    print(f"{label:20s} {path.exists()}  {path}")

In [ ]:
if not ANALYSIS_TRAJECTORY.exists():
    raise FileNotFoundError(
        "Missing analysis-ready trajectory. Run 08_trajectory_preprocessing.ipynb first. "
        f"Expected: {ANALYSIS_TRAJECTORY}"
    )

if not STRUCTURE.exists():
    raise FileNotFoundError(
        "Missing production TPR structure file needed for trajectory analysis. "
        f"Expected: {STRUCTURE}"
    )

## Load the preprocessed trajectory

The cell below is the analysis entry point. It deliberately loads the fitted trajectory from `analysis_ready/`, not the raw production trajectory.

In [ ]:
try:
    import mdtraj as md
except ImportError as exc:
    raise ImportError("Install the md optional dependencies to run analysis cells: pip install -e '.[md]'") from exc

trajectory = md.load(str(ANALYSIS_TRAJECTORY), top=str(STRUCTURE))
trajectory

RMSD, radius of gyration, and SASA cells should build from `trajectory` above. Advanced clustering and ML analysis are intentionally left out until the preprocessing layer is validated on production outputs.